# 🔵 04 — Clustering with K-Means & DBSCAN

Unsupervised learning on the **Mall Customer Segmentation** dataset (synthetic).

Covers: Elbow method · Silhouette score · K-Means · DBSCAN · PCA visualisation.

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

from visualizer import plot_clusters

# Synthetic customer data
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'age':              np.concatenate([np.random.normal(25,5,50), np.random.normal(40,8,80), np.random.normal(60,7,70)]),
    'annual_income':    np.concatenate([np.random.normal(30,5,50), np.random.normal(60,10,80), np.random.normal(85,12,70)]),
    'spending_score':   np.concatenate([np.random.normal(70,10,50), np.random.normal(50,15,80), np.random.normal(30,10,70)]),
})
print('Dataset shape:', df.shape)
df.describe()

In [ ]:
# --- Scale ---
scaler = StandardScaler()
X = scaler.fit_transform(df)

# --- Elbow method ---
inertias, silhouettes = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, 'bo-')
axes[0].set(title='Elbow Method', xlabel='k', ylabel='Inertia')
axes[1].plot(list(K_range), silhouettes, 'gs-')
axes[1].set(title='Silhouette Score', xlabel='k', ylabel='Score')
plt.tight_layout()
plt.show()
print(f'Best k by silhouette: {list(K_range)[np.argmax(silhouettes)]}')

In [ ]:
# --- K-Means with best k ---
best_k = list(K_range)[np.argmax(silhouettes)]
km = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
km_labels = km.fit_predict(X)

print(f'K-Means (k={best_k}) Silhouette: {silhouette_score(X, km_labels):.4f}')
plot_clusters(X, km_labels, centers=km.cluster_centers_,
              title=f'K-Means Clusters (k={best_k})')

In [ ]:
# --- DBSCAN ---
db = DBSCAN(eps=0.5, min_samples=5)
db_labels = db.fit_predict(X)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()
print(f'DBSCAN: {n_clusters} clusters, {n_noise} noise points')
if n_clusters > 1:
    print(f'Silhouette: {silhouette_score(X, db_labels):.4f}')
plot_clusters(X, db_labels, title='DBSCAN Clusters')